In [1]:
import os
os.environ["R_MAX_VSIZE"] = "100Gb"  # Increase R's maximum vector size if needed

# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
from rpy2.robjects.packages import importr
from rpy2.robjects import r, pandas2ri
import numpy as np

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import anndata2ri

anndata2ri.activate()
pandas2ri.activate()

# sys.path.append('../utils')
# from functions import * 

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

sns.set(style="whitegrid")

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis


/scratch/ipykernel_23623/160817418.py:29: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


scanpy==1.10.1 anndata==0.10.7 umap==0.5.6 numpy==1.26.4 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.4.2 statsmodels==0.14.2 igraph==0.11.5 louvain==0.8.2 pynndescent==0.5.12


In [2]:
# set up paths for files 
gene_exp = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/matrix.csv" #intronic and exonic reads 
ab_adata = sc.read_csv(gene_exp)
print(f"Done reading {gene_exp}")

# read in metadata
metadata = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/INFO/metadata.csv"
metadata = pd.read_csv(metadata)

WD="/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression"
today = datetime.datetime.now()
today = today.strftime("%Y-%m-%d")
ab_adata.var['gene_name'] = ab_adata.var_names
ab_adata.obs["source"] = "allen_brain"

# Ensure metadata matches the cells in adata
if 'sample_name' in metadata.columns:
    metadata.set_index('sample_name', inplace=True)
    ab_adata.obs = metadata.loc[ab_adata.obs_names]  # Align metadata to the cells in adata
else:
    raise ValueError("Metadata must have a column named 'sample_name' to match with gene expression data.")

# let's keep just the non-neuronal cells from ab_data 
ab_adata = ab_adata[ab_adata.obs["class_label"] == "Non-neuronal"].copy()
# rename ab_adata "gene_name" to "gene_symbol" and assess how many genes are in common between the two datasets
ab_adata.var.rename(columns={"gene_name":"gene_symbol"}, inplace=True)

Done reading /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/matrix.csv


In [3]:
# load in tabula sapien data 
tabsap_adata = sc.read_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TS_figshare/TabulaSapiens.h5ad")
# subset ts_adata to only cells from method=="smartseq2"
tabsap_adata = tabsap_adata[tabsap_adata.obs["method"]=="smartseq2"].copy()
tabsap_adata.obs["source"] = "tabula_sapiens"
print("Number of genes in common between the two datasets:", len(set(tabsap_adata.var["gene_symbol"]).intersection(set(ab_adata.var["gene_symbol"]))))

Number of genes in common between the two datasets: 30751


In [4]:
# This code is modified from the original from https://github.com/scverse/scanpy/issues/1643 

def run_sctransform(adata, layer=None, verbose=True, **kwargs):
    if layer and layer not in adata.layers:
        raise ValueError(f"Layer '{layer}' not found in AnnData object.")
    
    mat = adata.layers[layer] if layer else adata.X
    cell_names = adata.obs_names
    gene_names = adata.var_names

    # Assign to R
    r.assign('mat', mat.T)
    r.assign('cell_names', cell_names)
    r.assign('gene_names', gene_names)
    r('colnames(mat) <- cell_names')
    r('rownames(mat) <- gene_names')
    
    if verbose:
        print(f"Matrix set up for R: {len(gene_names)} genes, {len(cell_names)} cells.")

    # Seurat Object Creation
    seurat = importr('Seurat')
    r('seurat_obj <- CreateSeuratObject(mat)')
    
    if verbose:
        print("Seurat object created.")

    # SCTransform
    for k, v in kwargs.items():
        r.assign(k, v)
    kwargs_str = ', '.join([f'{k}={k}' for k in kwargs.keys()])
    r(f'seurat_obj <- SCTransform(seurat_obj, vst.flavor="v2", {kwargs_str})')

    # Handle gene filtering
    if verbose:
        print("Checking for genes filtered out by SCTransform.")
    filtout_genes = list(r('setdiff(rownames(mat), rownames(seurat_obj@assays$SCT@data))'))
    filtout_indicator = np.in1d(adata.var_names, filtout_genes)
    adata = adata[:, ~filtout_indicator]

    # Add SCT data back to AnnData
    if verbose:
        print("Adding SCT data back to AnnData.")
    sct_data = np.asarray(r['as.matrix'](r('seurat_obj@assays$SCT@data')))
    adata.layers['SCT_data'] = sct_data.T
    sct_counts = np.asarray(r['as.matrix'](r('seurat_obj@assays$SCT@counts')))
    adata.layers['SCT_counts'] = sct_counts.T

    if verbose:
        print("SCTransform completed and data added to AnnData")
        print(f"Final number of genes: {adata.n_vars} and cells: {adata.n_obs}")

    return adata

In [9]:
# add "raw_counts" layer to ab_adata
ab_adata.layers["raw_counts"] = ab_adata.X.copy()

In [6]:
tabsap_adata

AnnData object with n_obs × n_vars = 27051 × 58870
    obs: 'organ_tissue', 'method', 'donor', 'anatomical_information', 'n_counts_UMIs', 'n_genes', 'cell_ontology_class', 'free_annotation', 'manually_annotated', 'compartment', 'gender', 'source'
    var: 'gene_symbol', 'feature_type', 'ensemblid', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: '_scvi', '_training_mode', 'dendrogram_cell_type_tissue', 'dendrogram_computational_compartment_assignment', 'dendrogram_consensus_prediction', 'dendrogram_tissue_cell_type', 'donor_colors', 'donor_method_colors', 'hvg', 'method_colors', 'neighbors', 'organ_tissue_colors', 'sex_colors', 'tissue_colors', 'umap'
    obsm: 'X_pca', 'X_scvi', 'X_scvi_umap', 'X_umap'
    layers: 'decontXcounts', 'raw_counts'
    obsp: 'connectivities', 'distances'

In [10]:
ab_adata = run_sctransform(ab_adata, layer="raw_counts")
ab_adata.layers["SCT_data"]

Matrix set up for R: 50281 genes, 4753 cells.

    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

R[write to console]: Warning:
R[write to console]:  Data is of class matrix. Coercing to dgCMatrix.

R[write to console]: Running SCTransform on assay: RNA

R[write to console]: Running SCTransform on layer: counts



Seurat object created.


R[write to console]: vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

R[write to console]: Variance stabilizing transformation of count matrix of size 41115 by 4753

R[write to console]: Model formula is y ~ log_umi

R[write to console]: Get Negative Binomial regression parameters per gene

R[write to console]: Using 2000 genes, 4753 cells

R[write to console]: Error in getGlobalsAndPackages(expr, envir = envir, globals = globals) : 
  The total size of the 19 globals exported for future expression (‘FUN()’) is 594.17 MiB.. This exceeds the maximum allowed size of 500.00 MiB (option 'future.globals.maxSize'). The three largest globals are ‘FUN’ (575.56 MiB of class ‘function’), ‘umi_bin’ (18.26 MiB of class ‘numeric’) and ‘data_step1’ (274.20 KiB of class ‘list’)



RRuntimeError: Error in getGlobalsAndPackages(expr, envir = envir, globals = globals) : 
  The total size of the 19 globals exported for future expression (‘FUN()’) is 594.17 MiB.. This exceeds the maximum allowed size of 500.00 MiB (option 'future.globals.maxSize'). The three largest globals are ‘FUN’ (575.56 MiB of class ‘function’), ‘umi_bin’ (18.26 MiB of class ‘numeric’) and ‘data_step1’ (274.20 KiB of class ‘list’)


In [ ]:
tabsap_adata = run_sctransform(tabsap_adata, layer="raw_counts")
tabsap_adata.layers["SCT_data"]

In [ ]:
adata1 = ab_adata.copy()
adata2 = ts_adata.copy()

# Step 1: Align genes between the two datasets
common_genes = adata1.var["gene_symbol"].isin(adata2.var["gene_symbol"])
adata1 = adata1[:, common_genes]

common_genes = adata2.var["gene_symbol"].isin(adata1.var["gene_symbol"])
adata2 = adata2[:, common_genes]

# Ensure the genes are in the same order
adata2 = adata2[:, adata1.var_names]

# Step 2: Use raw counts from the respective layers
adata1_raw = adata1.layers["counts"]
adata2_raw = adata2.layers["raw_counts"]

# Step 3: Normalize raw counts
adata1.layers["normalized_raw"] = adata1_raw.copy() 
adata2.layers["normalized_raw"] = adata2_raw.copy()

sc.pp.normalize_total(adata1, layer="normalized_raw", target_sum=1e4)
sc.pp.log1p(adata1, layer="normalized_raw")

sc.pp.normalize_total(adata2, layer="normalized_raw", target_sum=1e4)
sc.pp.log1p(adata2, layer="normalized_raw")

# Replace .X with normalized raw counts
adata1.X = csr_matrix(adata1.layers["normalized_raw"])  # Ensure sparse matrix
adata2.X = csr_matrix(adata2.layers["normalized_raw"])  # Ensure sparse matrix

# Step 3: Combine datasets
adata1.obs["dataset"] = "AllenBrain"
adata2.obs["dataset"] = "TabulaSapien"

combined = adata1.concatenate(adata2, batch_key="dataset")
# need to add shared column in combined.obs for cell type... 
combined.obs["cell_type_combined"] = adata1.obs["subclass_label"].tolist() + adata2.obs["cell_ontology_class"].tolist()

In [ ]:
# get most variable genes 
sc.pp.highly_variable_genes(combined, n_top_genes=5000, flavor="seurat_v3", batch_key="dataset")

In [ ]:
# Step 4: Perform PCA and UMAP
sc.pp.pca(combined)
sc.pp.neighbors(combined)
sc.tl.umap(combined)

In [ ]:
combined.shape

In [ ]:
# I want to only color Microglia and Macrophage cells and monocyte 
cells_color = ["Microglia", "macrophage", "monocyte"]

combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined"].apply(
    lambda x: x if x in cells_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["source", "cell_type_combined_simplified"], palette="tab20",
    title="Combined Datasets UMAP (post Harmony!)",
)


In [ ]:
combined.obs.source.value_counts()

In [ ]:
combined.obs.subclass_label.value_counts()

In [ ]:
silhouette_batch = silhouette_score(combined.obsm["X_umap"], combined.obs["dataset"])
silhouette_biology = silhouette_score(combined.obsm["X_umap"], combined.obs["cell_type_combined_simplified"])
print(f"Silhouette Score by Batch: {silhouette_batch}")
print(f"Silhouette Score by Cell Type: {silhouette_biology}")

In [ ]:
sce.pp.harmony_integrate(combined, 'source')
# Check that Harmony has updated the PCA representation
print("Keys in obsm after Harmony:", combined.obsm.keys())

In [ ]:
# Compute neighbors and UMAP using the Harmony-corrected PCA
sc.pp.neighbors(combined, use_rep="X_pca_harmony")
sc.tl.umap(combined)

In [ ]:
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["source", "cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (post Harmony!)",
)


In [ ]:
# Get cell type counts and identify cell types with more than 1000 cells
cell_type_counts = combined.obs["cell_type_combined"].value_counts()
cell_types_to_color = cell_type_counts[cell_type_counts >= 500].index.tolist()

# Add a new "simplified" column where infrequent cell types are labeled as "Other"
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined"].apply(
    lambda x: x if x in cell_types_to_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")

# Define colors: assign distinct colors to frequent cell types, and use grey for "Other"
cell_type_colors = {
    cell_type: color for cell_type, color in zip(cell_types_to_color, sns.color_palette("husl", len(cell_types_to_color)))
}
cell_type_colors["Other"] = "grey"

# Update `uns` with the colors for visualization
combined.uns["cell_type_combined_simplified_colors"] = [
    cell_type_colors.get(ct, "grey") for ct in combined.obs["cell_type_combined_simplified"].cat.categories
]

# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (Simplified)",
)


In [ ]:
# Define a function to clean up cell type labels
def clean_cell_type(cell_type):
    if "endothelial" in cell_type.lower():
        return "Endothelial"
    elif "b cell" in cell_type.lower():
        return "B cell"
    elif "t cell" in cell_type.lower():
        return "T cell"
    elif "stem" in cell_type.lower():
        return "Stem cell"
    elif "monocyte" in cell_type.lower():
        return "Monocyte"
    else:
        return cell_type  # Keep the original label if no rule matches
    
combined.obs["clean_cell_type"] = combined.obs["cell_type_combined"].apply(clean_cell_type)


In [ ]:
# Get cell type counts and identify cell types with more than 1000 cells
cell_type_counts = combined.obs["clean_cell_type"].value_counts()
cell_types_to_color = cell_type_counts[cell_type_counts >= 750].index.tolist()

# Add a new "simplified" column where infrequent cell types are labeled as "Other"
combined.obs["cell_type_combined_simplified"] = combined.obs["clean_cell_type"].apply(
    lambda x: x if x in cell_types_to_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")

# Define colors: assign distinct colors to frequent cell types, and use grey for "Other"
cell_type_colors = {
    cell_type: color for cell_type, color in zip(cell_types_to_color, sns.color_palette("husl", len(cell_types_to_color)))
}
cell_type_colors["Other"] = "grey"

# Update `uns` with the colors for visualization
combined.uns["cell_type_combined_simplified_colors"] = [
    cell_type_colors.get(ct, "grey") for ct in combined.obs["cell_type_combined_simplified"].cat.categories
]

# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (Simplified)",
)


In [ ]:
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["organ_tissue"])

In [ ]:
combined[combined.obs["cell_type_combined_simplified"] == "macrophage"]

In [ ]:
# plot only macrophage cell type on umap
sc.pl.umap(combined[combined.obs["cell_type_combined_simplified"] == "macrophage"], color="organ_tissue")

In [ ]:
combined.obs.organ_tissue.value_counts()

In [ ]:
combined.obs[combined.obs["clean_cell_type"] == "T cell"].dataset.value_counts()

In [ ]:
combined.obs[combined.obs["clean_cell_type"] == "macrophage"].dataset.value_counts()

In [ ]:
# save combined data 
!pwd

In [ ]:
combined.var

In [ ]:
if "cell_type_combined_simplified_colors" in combined.uns:
    del combined.uns["cell_type_combined_simplified_colors"]

In [ ]:
today = datetime.date.today()
today = today.strftime("%Y-%m-%d")

# save file
combined.write_h5ad(f"/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/TabulaSapien_AllenBrain_combined_nonneuron_adata_{today}.h5ad")